In [ ]:
# 🛠️ Setup: install deps + fetch the shared helper on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformer_lens datasets transformers plotly
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/tiny.py
import tiny
import torch
import plotly.graph_objects as go

torch.manual_seed(42)
print("device:", tiny.device())

<div dir="rtl">

> 🧪 **نسخة التمرين.** الخلايا اللي فيها `# TODO` ناقصة عمدًا — املأها بنفسك.
> لو علقت، النسخة المرجعية (`..._reference.ipynb`) فيها الحل كامل.

</div>

<div dir="rtl">

# المرحلة ٢أ: أول كتلة انتباه — لما الكلمة تقدر تبص لجيرانها

في المرحلة ١ج بنينا نموذج "تضمينات بس" (embeddings only): كل وحدة فرعية (subword) ليها متجه ثابت، والنموذج بيتنبأ بالكلمة اللي جاية من غير ما يبص لأي كلمة تانية. ده نموذج **أعمى للسياق** (context-blind).

النهارده بنضيف حاجة واحدة بس: **كتلة انتباه واحدة برأس واحد** (one attention block, one head). الإضافة دي بتدي الموديل قدرة جديدة: الكلمة تقدر **تبص للكلمات اللي قبلها** وتستعمل المعلومة دي وهي بتتنبأ.

**حدود التجربة (مهم):** الموديل ده صغير جدًا واتدرب على بيانات قليلة، فالقيم اللي هتطلع هتبان ضعيفة أو شبه عشوائية — وده طبيعي. هدفنا مش نبني موديل شاطر؛ هدفنا **نشوف شكل الانتباه وإزاي نقرأه**.

</div>

<div dir="ltr">

# Stage 2a: The first attention block — when a token can look at its neighbours

Stage 1c gave us an *embeddings-only* model: every subword has a fixed vector, and the model predicts the next token **without looking at any other token**. That model is **context-blind**.

Today we add exactly one thing: **one attention block with one head**. That unlocks a new capability — a token can **look at the tokens before it** and use that when predicting.

**Limits (important):** this model is tiny and trained on little data, so the values will look weak or near-random — that's fine. The goal isn't a smart model, it's to *see the shape of attention and learn how to read it*.

</div>

<div dir="rtl">

## ١. البيانات · Data

هنستعمل نفس مصادر المرحلة ١ج: نصوص فصحى (MSA) من ويكيبيديا، ونصوص مصري (Masri) من تويتات.

</div>

In [ ]:
# 📦 Fetch MSA + Masri text (same sources as Stage 1c)
from datasets import load_dataset
import re

MAX_CHARS = globals().get("MAX_CHARS", 500_000)

print("Fetching MSA from Wikipedia and Masri from Tweets...")
msa_stream = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train", streaming=True)
tweets_ds = load_dataset("amgadhasan/arabic_tweets_dialects", split="train")
eg_tweets = tweets_ds.filter(lambda x: x["dialect"] == "EG")

def clean(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[a-zA-Z0-9_@]+", "", text)
    return re.sub(r"[^\s\u0621-\u064A]", "", text)

def collect(rows, key, max_chars):
    out = ""
    for r in rows:
        out += clean(r[key]) + " "
        if len(out) >= max_chars:
            break
    return out[:max_chars]

msa_text = collect(msa_stream, "text", MAX_CHARS)
masri_text = collect(eg_tweets, "text", MAX_CHARS)
print(f"MSA chars: {len(msa_text)}  |  Masri chars: {len(masri_text)}")

<div dir="rtl">

## ٢. مين هيقطّع الكلام؟ · Which tokenizer?

في المرحلة ١ج درّبنا مُجزّئ (tokenizer) صغير من الصفر، فكان بيكسّر الكلمات لحروف مالهاش معنى. دلوقتي هنستعمل مُجزّئ **مُدرّب مسبقًا** عشان التوكنز تبقى مقروءة. هنقارن اتنين:

- **mGPT**: بيفصل أداة التعريف **"ال"** كتوكن لوحدها (الكتاب ← ال · كتاب). كويس عشان نشوف الانتباه بيربط بين أجزاء ليها معنى.
- **MARBERT**: متدرّب على تويتات عربي فيها مصري، فبيدّي كل كلمة مصري **توكن واحد كامل** (دلوقتي، اللي، عايز). أوضح للقراءة بس مبيفصلش "ال".

</div>

In [ ]:
from transformers import AutoTokenizer

sample = ["السمك", "الكتاب", "دلوقتي", "اللي", "عايز"]
mgpt = AutoTokenizer.from_pretrained("ai-forever/mGPT")
marbert = AutoTokenizer.from_pretrained("UBC-NLP/MARBERT")

def split(tok, w):
    return [tok.decode([i]).strip() for i in tok.encode(w, add_special_tokens=False)]

print(f"{'word':10} | {'mGPT':30} | MARBERT")
print("-" * 60)
for w in sample:
    print(f"{w:10} | {str(split(mgpt, w)):30} | {split(marbert, w)}")

<div dir="rtl">

## ٣. نجهّز بيانات التدريب · Build the training data

هنتدرّب على تقطيع **mGPT** (لأنه بيفصل "ال"). بس قاموسه ضخم (١٠٠ ألف توكن)، وموديل صغير ميقدرش يشيل ده. الحل: ناخد بس التوكنز اللي **ظهرت فعلًا** في نصوصنا ونرقّمها من جديد في مساحة صغيرة — كده الموديل يفضل صغير وقابل للتدريب. الدالة `tiny.make_compact_encoder` بتعمل ده.

</div>

In [ ]:
# Keep only the subwords that appear, remapped to a small dense vocab.
encode, corpus_ids, id_to_str, VOCAB = tiny.make_compact_encoder(mgpt, [msa_text, masri_text])
all_ids = corpus_ids[0] + corpus_ids[1]
print(f"compact vocab: {VOCAB} subwords  |  tokens: {len(all_ids)}")  # ← (n_ids,)

<div dir="rtl">

## ٤. نبني الموديل وندرّبه · Build & train

موديل بطبقة واحدة (`n_layers=1`) ورأس انتباه واحد (`n_heads=1`). الفرق الوحيد عن المرحلة ١ج هو **كتلة الانتباه** دي.

</div>

In [ ]:
N_CTX = globals().get("N_CTX", 64)
N_EPOCHS = globals().get("N_EPOCHS", 200)

batches = tiny.make_natural_batches(all_ids, n_ctx=N_CTX)
print("batches:", tuple(batches.shape))  # ← (N, n_ctx)

# TODO: build a ONE-layer, ONE-head model — the single attention block this
# notebook is about. Fill in n_layers and n_heads.
model = tiny.make_tiny_model(n_layers=..., n_heads=..., d_vocab=VOCAB, n_ctx=N_CTX)  # TODO
losses = tiny.train(model, batches, n_epochs=N_EPOCHS, log_every=20)
print("loss:", round(losses[0], 3), "->", round(losses[-1], 3))  # ← scalar

<div dir="rtl">

## ٥. بإيدينا · By hand

الانتباه مش صندوق أسود. هنحسب نمط الانتباه لجملة واحدة **بإيدينا** بضرب الاستعلام (Query, `q`) في المفتاح (Key, `k`) وبعدين softmax، وبعدها نثبت إن النتيجة بتطابق اللي الموديل بيطلعه من جوّاه — الفرق صفر.

</div>

In [ ]:
prompt = "القطة بتاكل السمك"
comp = encode(prompt)
ids = torch.tensor([comp]).to(tiny.device())
str_toks = [id_to_str[i] for i in comp]
print("tokens:", str_toks)  # ← (seq,)

_, cache = model.run_with_cache(ids)
q = cache["q", 0][0, :, 0, :]                 # ← (seq, d_head)
k = cache["k", 0][0, :, 0, :]                 # ← (seq, d_head)

# TODO: compute the attention pattern BY HAND, then prove it equals the model's.
#   1) scores = (q @ k.T) / sqrt(d_head)
#   2) mask the upper triangle with -inf (no peeking at the future)
#   3) by_hand = softmax over the last dim
scores = ...   # TODO  (hint: q @ k.T / (q.shape[-1] ** 0.5))
by_hand = ...  # TODO  (hint: torch.softmax(scores.masked_fill(mask, float("-inf")), dim=-1))

from_cache = cache["pattern", 0][0, 0]        # ← (seq, seq)
print("max abs diff (by-hand vs model):", float((by_hand - from_cache).abs().max()))  # ← should be ~0

<div dir="rtl">

## ٦. الخريطة الحرارية · The heatmap

الرقم فوق قريب من الصفر، يبقى الصورة اللي تحت **هي** مصفوفة الانتباه نفسها. (بنعرض من اليمين للشمال زي العربي.)

</div>

In [ ]:
def show_attention(model, prompt_ids, str_tokens, layer=0, head=0, title=""):
    _, cache = model.run_with_cache(prompt_ids)
    pattern = cache["pattern", layer][0, head].detach().cpu().numpy()  # ← (seq, seq)
    fig = go.Figure(go.Heatmap(z=pattern, x=str_tokens, y=str_tokens, colorscale="Blues"))
    fig.update_layout(title=title, xaxis=dict(side="top"))
    fig.update_xaxes(autorange="reversed")  # RTL: first Arabic token on the right
    fig.update_yaxes(autorange="reversed")
    return fig

show_attention(model, ids, str_toks, 0, 0, "Head 0 · القطة بتاكل السمك").show()

<div dir="rtl">

## ٧. نقرأ الأرقام · Reading the values

كل خانة في الخريطة رقم بين ٠ و ١: **قد إيه** التوكن اللي في الصف ده بيدّي انتباه للتوكن اللي في العمود. وفيه قاعدتين دايمًا صح، حتى لو الموديل لسه بيتعلم:

1. **كل صف بيجمع ١**، لأنه توزيع احتمالي — التوكن بيوزّع انتباهه كله على اللي قبله.
2. **النص اللي فوق القطر فاضي (أصفار)**، لأن الموديل **مايقدرش يبص للمستقبل** (causal mask): كل توكن يشوف اللي قبله بس.

أما **قيمة** كل خانة بالظبط فهي ضعيفة/شبه متساوية هنا لأن الموديل صغير واتدرب شوية — ده متوقع. المهم دلوقتي هو **الشكل** (المثلث السببي + كل صف بيجمع ١)، مش الأرقام نفسها.

</div>

In [ ]:
row_sums = from_cache.sum(dim=-1)
print("row sums (each token's attention adds to 1):",
      [round(x, 3) for x in row_sums.tolist()])

above_diagonal = torch.triu(from_cache, diagonal=1)
print("everything above the diagonal is 0 (no peeking at the future):",
      bool((above_diagonal == 0).all()))

<div dir="rtl">

## ٨. هل السياق بيغيّر التوقع؟ · Does context change the prediction?

ده **بيت القصيد** من إضافة الانتباه. هنبني **شجرة توقعات**: من سياق مُعطى، ناخد أعلى احتمالات للـ subword الجاية، ونغذّي الموديل توقعه ونكمّل خطوة خطوة. (السهم: من توكن للـ subword المتوقع بعده · سُمكه: الاحتمال · الرقم: الاحتمال نفسه. أحمر = **مالهاش معنى**: `[UNK]` أو احتمال ضعيف `< 0.10`.)

**الأساس — نموذج التضمينات (١ج)، أعمى للسياق:** الصورة تحت جملتين مختلفتين بينتهوا بنفس الكلمة (السمك). نموذج التضمينات بيدّي **نفس** الشجرة بالظبط — لأنه بيبص للتوكن الأخير بس، ومش شايف اللي قبله.

</div>

<div dir="rtl">

![embeddings-only is context-blind](images/embeddings_context_blind.png)

*(صورة محفوظة من نموذج التضمينات — مش بندرّبه هنا تاني. الشجرتين متطابقتين = السياق مُتجاهَل.)*

</div>

In [ ]:
LOW_PROB = 0.10  # below this, the model is basically guessing -> flag it (red)

def next_topk(context_ids, k):
    ids = torch.tensor([context_ids]).to(tiny.device())
    probs = torch.softmax(model(ids)[0, -1], dim=-1)
    vals, idx = torch.topk(probs, k)
    return [(int(i), float(v)) for v, i in zip(vals, idx)]

def build_prediction_tree(context_text, max_depth=2, top_k=2):
    """Grow a tree from `context_text` by feeding the attention model its own
    top-k predictions step by step. Every node is conditioned on the FULL path
    (the context) — which is exactly what attention made possible.

    Returns edges [(parent_key, child_key, prob, depth, sensible)] and
    nodes {key: (label, sensible)}; `sensible` is False for [UNK] or prob<LOW_PROB.
    """
    edges, nodes = [], {"root": (context_text, True)}
    frontier, n = [("root", encode(context_text), 0)], 0
    while frontier:
        pkey, ctx, depth = frontier.pop(0)
        if depth >= max_depth:
            continue
        for tid, p in next_topk(ctx, top_k):
            n += 1
            label = id_to_str.get(tid, "[UNK]")
            sensible = (label != "[UNK]") and (p >= LOW_PROB)
            ckey = f"{depth}:{n}:{label}"
            nodes[ckey] = (label, sensible)
            edges.append((pkey, ckey, p, depth, sensible))
            frontier.append((ckey, ctx + [tid], depth + 1))
    return edges, nodes

# Same two contexts as the baseline image: different prefix, same last word.
CONTEXTS = ["القطة بتاكل السمك", "الولد بياكل السمك"]
trees = [build_prediction_tree(c, max_depth=2, top_k=2) for c in CONTEXTS]

In [ ]:
from collections import defaultdict
from plotly.subplots import make_subplots

def tree_layout(edges):
    depth_of = {"root": 0}
    for _pk, ck, _p, d, _s in edges:
        depth_of[ck] = d + 1
    levels = defaultdict(list)
    for k, d in depth_of.items():
        levels[d].append(k)
    return {k: (d, (len(ks) - 1) / 2 - i)
            for d, ks in levels.items() for i, k in enumerate(ks)}

def add_tree(fig, col, edges, nodes):
    sfx = "" if col == 1 else str(col)
    xref, yref = f"x{sfx}", f"y{sfx}"
    pos = tree_layout(edges)
    for pk, ck, p, _d, s in edges:
        (x0, y0), (x1, y1) = pos[pk], pos[ck]
        color = "#c0392b" if not s else "#2b6cb0"  # red = doesn't make sense
        fig.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref=xref, yref=yref, axref=xref,
                           ayref=yref, showarrow=True, arrowhead=3, arrowwidth=1.5 + p * 7.5,
                           arrowcolor=color, standoff=18, startstandoff=24, text="")
        fig.add_annotation(x=(x0 + x1) / 2, y=(y0 + y1) / 2, xref=xref, yref=yref,
                           showarrow=False, text=f"{p:.2f}", font=dict(size=10, color="#333"),
                           bgcolor="rgba(255,255,255,0.75)")
    keys = list(pos.keys())
    fig.add_trace(go.Scatter(
        x=[pos[k][0] for k in keys], y=[pos[k][1] for k in keys], mode="markers+text",
        text=[nodes[k][0] for k in keys], textposition="middle center",
        textfont=dict(size=13, color="#111"),
        marker=dict(color=["#f6c350" if k == "root" else ("#f6c0bb" if not nodes[k][1] else "#d9d9d9")
                           for k in keys],
                    size=[40 if k == "root" else 30 for k in keys], line=dict(width=1.5, color="#333")),
        hoverinfo="text", showlegend=False), row=1, col=col)
    fig.update_xaxes(visible=False, row=1, col=col)
    fig.update_yaxes(visible=False, row=1, col=col)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=[f"السياق: {c}" for c in CONTEXTS])
for i, (edges, nodes) in enumerate(trees):
    add_tree(fig, i + 1, edges, nodes)
fig.update_layout(height=460, margin=dict(l=20, r=20, t=80, b=20),
                  title_text="نموذج الانتباه: السياق بيغيّر التوقع · attention model — context changes the prediction")
fig.show()

<div dir="rtl">

## ٩. الخلاصة والخطوة الجاية · Recap & handoff

- ضفنا **كتلة انتباه واحدة برأس واحد** فوق التضمينات.
- استعملنا مُجزّئ مُدرّب (mGPT) عشان التوكنز تبقى مقروءة، مع تصغير القاموس عشان الموديل يفضل قابل للتدريب.
- حسبنا نمط الانتباه **بإيدينا** وأثبتنا إنه نفس اللي الموديل بيعمله (الفرق صفر).
- اتعلمنا **نقرأ** الخريطة: كل صف بيجمع ١، والمستقبل مقفول (المثلث السببي).
- شفنا بـ**شجرة التوقعات** إن السياق **بيغيّر** التوقع (نموذج الانتباه)، عكس نموذج التضمينات الأعمى للسياق — وده اللي الانتباه ضافه. (أحمر = توقع مالوش معنى؛ موديل صغير فطبيعي يبقى فيه كتير.) هنعيد نفس الشجرة في المراحل الجاية ونشوفها بتتحسّن مع كل إضافة.

**المرحلة الجاية (٢ب):** نضيف **كذا رأس** (multiple heads) في نفس الطبقة، ونشوف كل رأس بيتخصص في علاقة مختلفة — الموديل يمسك أكتر من علاقة في نفس الوقت.

</div>

<div dir="ltr">

## 9. Recap & handoff

- We added **one attention block with one head** on top of the embeddings.
- We used a **pretrained tokenizer (mGPT)** for readable tokens, with a compact remap so the model stays trainable.
- We computed the attention pattern **by hand** and proved it equals the model's own (diff ≈ 0).
- We learned to **read** the map: every row sums to 1, and the future is masked (the causal triangle).
- The **prediction tree** showed that context **changes** the prediction (attention model), unlike the context-blind embeddings model — that's what attention bought us. (Red = nonsense; expected on a tiny model.) We'll regrow the same tree in later rungs to watch it improve.

**Next (2b):** add **multiple heads** in the same layer and watch each head specialise on a different Arabic relation — several relations captured at once.

</div>